In [1]:
from core.config import CURRENT
from core.clean import clean_text
from core.models import get_svm

import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from nltk.corpus import stopwords

# SVM on 20K rows
df = pd.read_csv(CURRENT["path"])
df = df.dropna(subset=[CURRENT["label_col"]])
df = df.sample(n=20000, random_state=42).reset_index(drop=True)
df["Text"] = df[CURRENT["text_col"]].apply(clean_text)

train, test = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df[CURRENT["label_col"]]
)

vect = TfidfVectorizer(
    stop_words=stopwords.words('german'),
    max_features=CURRENT["max_features"],
)
X_train = vect.fit_transform(train["Text"])
X_test = vect.transform(test["Text"])

y_train = train[CURRENT["label_col"]]
y_test = test[CURRENT["label_col"]]

model = get_svm()

start = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start

y_pred = model.predict(X_test)

labels = sorted(set(y_test))
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=labels, zero_division=0
)

df_report = pd.DataFrame({
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "Support": sup,
}, index=[int(l) for l in labels])

print(f"Train time: {train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(df_report.round(4))

Train time: 76.80s
Accuracy: 0.8337
F1 macro: 0.3232

   Precision  Recall      F1  Support
1     0.9017  0.9760  0.9374     3252
2     0.2500  0.1095  0.1523      210
3     0.0000  0.0000  0.0000       78
4     0.2692  0.0609  0.0993      115
5     0.3320  0.4175  0.3699      194
6     0.4464  0.3311  0.3802      151
